# 单策略个图（Plotly交互版）

按原代码连续展示方式绘图，同时保留 Plotly 交互能力。


In [25]:
# 导入数据处理库
import pandas as pd

# 导入日期处理
from datetime import datetime

# 导入Plotly图形对象
import plotly.graph_objects as go

# 导入Plotly双Y轴子图工具
from plotly.subplots import make_subplots

# 导入项目内策略数据函数
import get_data_order as gdo

# 导入交易日回溯函数
from ratespricer import d_get_bus_day


In [ ]:
# ***** 可修改参数开始 *****

# 配置文件路径
FILE_PATH = 'report.xlsx'

# 策略名: 基差/利差/跨期/期货利差/IRS利差/现券利差
STRATEGY = '期货利差'

# 标的1代码
CODE1 = 'T2603'

# 标的2代码
CODE2 = 'TF2609'

# 比率参数(主要用于基差)
RATIO = 'cf'

# 跨期策略使用的spread参数
SPREAD = 20

# 结束日期
END_DATE = datetime.now().date()
END_DATE = datetime.strptime('2026-03-09', '%Y-%m-%d').date()

# 回溯交易日天数
LOOKBACK_BUS_DAYS = 60

# ***** 可修改参数结束 *****


In [27]:
# 计算开始日期
START_DATE = d_get_bus_day(END_DATE, -LOOKBACK_BUS_DAYS)

# 打印时间区间与参数
print(f'Start: {START_DATE}, End: {END_DATE}')
print(f'STRATEGY: {STRATEGY}')
print(f'CODE1: {CODE1}')
print(f'CODE2: {CODE2}')

# 读取最廉券配置
def load_ctd_config(file_path):
    ctd_config = pd.read_excel(file_path, sheet_name='最廉券配置')
    return ctd_config

# 规范债券代码
def as_bond_code(code):
    return str(int(code))

# 查询期货对应CTD
def safe_ctd_code(ctd_config, fut_code):
    ctd_code = ctd_config.loc[ctd_config['Fut'] == fut_code, 'Spot'].values[0]
    return str(int(ctd_code))

# 统一关键列类型
def normalize_df(df):
    data = df.copy()
    data['datetime'] = pd.to_datetime(data['datetime'], errors='coerce')
    data = data.dropna(subset=['datetime']).sort_values('datetime')
    if 'difference' in data.columns:
        data['difference'] = pd.to_numeric(data['difference'], errors='coerce')
    if 'irr' in data.columns:
        data['irr'] = pd.to_numeric(data['irr'], errors='coerce')
    # 按原代码思路转为字符串横轴，图像连续展示
    data['x_plot'] = data['datetime'].dt.strftime('%Y-%m-%d %H:%M')
    return data

# 按策略获取绘图数据和标题
def build_strategy_df_and_title(strategy, code1, code2, ratio, spread, start_date, end_date, ctd_config):
    target_str = gdo.get_t_str(start_date, end_date)

    if strategy == '基差':
        df = pd.DataFrame(gdo.get_basis(as_bond_code(code1), str(code2), start_date, end_date, ratio))
        df.columns = ['datetime', 'difference', 'y1', 'y2']
        df_irr = pd.DataFrame(gdo.get_basis_irr(as_bond_code(code1), str(code2), start_date, end_date, ratio))
        df_irr.columns = ['datetime', 'irr']
        data = pd.merge(df, df_irr, on='datetime').sort_values('datetime')
        title = f"{as_bond_code(code1)}_{str(code2)}_{target_str}"
        return normalize_df(data), title

    if strategy == '利差':
        ctd_code = safe_ctd_code(ctd_config, str(code2))
        data = pd.DataFrame(gdo.get_bond_fut_spread(str(code2), ctd_code, as_bond_code(code1), start_date, end_date))
        data.columns = ['datetime', 'fut_rate', 'bond', 'difference']
        data = data.sort_values('datetime')
        title = f"{as_bond_code(code1)}_{target_str}"
        return normalize_df(data), title

    if strategy == '跨期':
        fut_code_1 = str(code1)
        fut_code_2 = str(code2)
        ctd_code_1 = safe_ctd_code(ctd_config, fut_code_1)
        ctd_code_2 = safe_ctd_code(ctd_config, fut_code_2)
        df = pd.DataFrame(gdo.get_spread_futures(fut_code_1, fut_code_2, start_date, end_date, 1))
        df.columns = ['datetime', 'difference', 'y1', 'y2']
        df_irr = pd.DataFrame(gdo.get_fut_fut_irr(fut_code_1, ctd_code_1, fut_code_2, ctd_code_2, spread, start_date, end_date))
        df_irr.columns = ['datetime', 'irr']
        data = pd.merge(df, df_irr, on='datetime').sort_values('datetime')
        title = f"{fut_code_1}_{fut_code_2}_{target_str}"
        return normalize_df(data), title

    if strategy == '期货利差':
        ctd_code_1 = safe_ctd_code(ctd_config, str(code1))
        ctd_code_2 = safe_ctd_code(ctd_config, str(code2))
        data = pd.DataFrame(gdo.get_fut_fut_spread(str(code1), ctd_code_1, str(code2), ctd_code_2, start_date, end_date))
        data.columns = ['datetime', 'difference', 'y1', 'y2']
        data = data.sort_values('datetime')
        title = f"{str(code1)}_{str(code2)}_{target_str}"
        return normalize_df(data), title

    if strategy == 'IRS利差':
        ctd_code = safe_ctd_code(ctd_config, str(code2))
        data = pd.DataFrame(gdo.get_irs_fut_spread(str(code1), str(code2), ctd_code, start_date, end_date))
        data.columns = ['datetime', 'difference', 'y1', 'y2']
        data = data.sort_values('datetime')
        title = f"{str(code1)}_{str(code2)}_{target_str}"
        return normalize_df(data), title

    if strategy == '现券利差':
        data = pd.DataFrame(gdo.get_spread_spot(as_bond_code(code1), as_bond_code(code2), start_date, end_date))
        data.columns = ['datetime', 'difference', 'y1', 'y2']
        data = data.sort_values('datetime')
        title = f"{as_bond_code(code1)}_{as_bond_code(code2)}_{target_str}"
        return normalize_df(data), title

    raise ValueError(f'不支持的策略: {strategy}')

# 添加分位线与极值线
def add_stat_lines(fig, data):
    diff = pd.to_numeric(data['difference'], errors='coerce').dropna()
    if diff.empty:
        return
    q25 = diff.quantile(0.25)
    q50 = diff.quantile(0.50)
    q75 = diff.quantile(0.75)
    min_val = diff.min()
    max_val = diff.max()

    fig.add_hline(y=max_val, line_dash='dash', line_color='black', line_width=1)
    fig.add_hline(y=q75, line_dash='dash', line_color='black', line_width=1)
    fig.add_hline(y=q50, line_dash='solid', line_color='black', line_width=1)
    fig.add_hline(y=q25, line_dash='dash', line_color='black', line_width=1)
    fig.add_hline(y=min_val, line_dash='dash', line_color='black', line_width=1)

    fig.add_annotation(x=1.01, y=max_val, xref='paper', yref='y', text=f'max {max_val:.3f}', showarrow=False)
    fig.add_annotation(x=1.01, y=q75, xref='paper', yref='y', text=f'75% {q75:.3f}', showarrow=False)
    fig.add_annotation(x=1.01, y=q50, xref='paper', yref='y', text=f'50% {q50:.3f}', showarrow=False)
    fig.add_annotation(x=1.01, y=q25, xref='paper', yref='y', text=f'25% {q25:.3f}', showarrow=False)
    fig.add_annotation(x=1.01, y=min_val, xref='paper', yref='y', text=f'min {min_val:.3f}', showarrow=False)

# 添加最大最小点
def add_extreme_markers(fig, data):
    clean_data = data.copy()
    clean_data['difference'] = pd.to_numeric(clean_data['difference'], errors='coerce')
    clean_data = clean_data.dropna(subset=['difference'])
    if clean_data.empty:
        return
    max_row = clean_data.loc[clean_data['difference'].idxmax()]
    min_row = clean_data.loc[clean_data['difference'].idxmin()]

    fig.add_trace(
        go.Scatter(
            x=[max_row['x_plot']],
            y=[max_row['difference']],
            mode='markers',
            name='max',
            marker=dict(color='red', size=8),
            hovertemplate='max: %{y:.3f}<extra></extra>'
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=[min_row['x_plot']],
            y=[min_row['difference']],
            mode='markers',
            name='min',
            marker=dict(color='green', size=8),
            hovertemplate='min: %{y:.3f}<extra></extra>'
        ),
        secondary_y=False
    )

# 生成自适应横坐标刻度
def build_adaptive_ticks(plot_data, max_labels=10):
    d = plot_data[['datetime', 'x_plot']].drop_duplicates().sort_values('datetime').reset_index(drop=True)
    if d.empty:
        return [], []

    n = len(d)
    max_labels = max(6, max_labels)

    # 关键点: 首尾 + 每日首尾 + 每日关键时点
    cand = set([0, n - 1])
    by_day = d.groupby(d['datetime'].dt.date)
    key_times = ['09:30', '10:30', '11:30', '13:00', '14:00', '15:00']

    for day, g in by_day:
        idxs = g.index.tolist()
        cand.add(idxs[0])
        cand.add(idxs[-1])
        for t in key_times:
            target = pd.to_datetime(f'{day} {t}')
            near_idx = (d.loc[idxs, 'datetime'] - target).abs().idxmin()
            cand.add(int(near_idx))

    # 不足时按均匀间隔补点
    if len(cand) < max_labels:
        step = max(1, int(n / max_labels))
        for i in range(0, n, step):
            cand.add(i)

    idx = sorted(cand)

    # 过多时压缩到 max_labels
    if len(idx) > max_labels:
        pick = [idx[0]]
        if max_labels > 2:
            stride = (len(idx) - 1) / (max_labels - 1)
            for k in range(1, max_labels - 1):
                pick.append(idx[int(round(k * stride))])
        pick.append(idx[-1])
        idx = sorted(set(pick))

    tickvals = d.loc[idx, 'x_plot'].tolist()
    one_day = d['datetime'].dt.date.nunique() == 1
    if one_day:
        ticktext = d.loc[idx, 'datetime'].dt.strftime('%H:%M').tolist()
    else:
        ticktext = d.loc[idx, 'datetime'].dt.strftime('%m-%d %H:%M').tolist()

    return tickvals, ticktext

# 使用Plotly绘制交互图
def plot_interactive(data, title):
    plot_data = data.copy()
    plot_data['difference'] = pd.to_numeric(plot_data['difference'], errors='coerce')
    plot_data = plot_data.dropna(subset=['difference'])
    if plot_data.empty:
        raise ValueError('difference 列没有可用的数值数据，请检查数据源或参数')
    if 'irr' in plot_data.columns:
        plot_data['irr'] = pd.to_numeric(plot_data['irr'], errors='coerce')

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    fig.add_trace(
        go.Scatter(
            x=plot_data['x_plot'],
            y=plot_data['difference'],
            mode='lines',
            name='difference',
            line=dict(color='midnightblue', width=2),
            hovertemplate='difference: %{y:.3f}<extra></extra>'
        ),
        secondary_y=False
    )

    has_irr = 'irr' in plot_data.columns and plot_data['irr'].notna().any()
    if has_irr:
        fig.add_trace(
            go.Scatter(
                x=plot_data['x_plot'],
                y=plot_data['irr'],
                mode='lines',
                name='irr',
                line=dict(color='red', width=1.5),
                hovertemplate='irr: %{y:.3f}<extra></extra>'
            ),
            secondary_y=True
        )

    add_stat_lines(fig, plot_data)
    add_extreme_markers(fig, plot_data)

    tickvals, ticktext = build_adaptive_ticks(plot_data, max_labels=10)

    fig.update_layout(
        title=title,
        template='plotly_white',
        height=700,
        hovermode='x unified',
        dragmode='zoom',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
    )

    # 使用分类横轴，按原代码风格连续展示
    fig.update_xaxes(
        title_text='datetime',
        type='category',
        tickmode='array',
        tickvals=tickvals,
        ticktext=ticktext,
        showspikes=True,
        spikemode='across',
        spikesnap='cursor'
    )

    fig.update_yaxes(title_text='difference', secondary_y=False)
    if has_irr:
        fig.update_yaxes(title_text='irr', secondary_y=True, showgrid=False, autorange='reversed')

    fig.show(config={'scrollZoom': True, 'displaylogo': False})

# 读取CTD配置
ctd_cfg = load_ctd_config(FILE_PATH)

# 获取数据与标题
df_plot, plot_title = build_strategy_df_and_title(
    STRATEGY, CODE1, CODE2, RATIO, SPREAD, START_DATE, END_DATE, ctd_cfg
)

# 绘制交互图
plot_interactive(df_plot, plot_title)


Start: 2026-01-30 00:00:00, End: 2026-03-09
STRATEGY: 利差
CODE1: 2500006
CODE2: TL2606
